# Notebook 3 — NL2SQL2NL (Linguagem Natural → SQL → Linguagem Natural)

Este notebook implementa o fluxo completo NL2SQL2NL:
1. Recebe uma pergunta em português
2. Gera SQL usando LLM com contexto do dicionário de dados
3. Executa a SQL no Postgres
4. Converte o resultado em resposta em linguagem natural

O fluxo é simples, sem LangGraph, usando apenas chamadas diretas com LangChain e OpenAI.

In [1]:
import os
import sys
import yaml
import logging
from pathlib import Path

print(f"Diretório de trabalho: {os.getcwd()}")

# Adiciona o diretório src ao path para importar os módulos do projeto
projeto_raiz = Path(os.getcwd()).parent
sys.path.insert(0, str(projeto_raiz / "src"))
print(f"PYTHONPATH: {sys.path[0]}")

Diretório de trabalho: /home/jovyan/work/notebooks
PYTHONPATH: /home/jovyan/work/src


In [2]:
# Importa módulos do projeto
from nl2sql2nl.config import OPENAI_API_KEY, POSTGRES_HOST, POSTGRES_PORT, POSTGRES_DB
from nl2sql2nl.database import executar_consulta
from nl2sql2nl.llm import chamar_modelo
from nl2sql2nl.logger import obter_logger

logger = obter_logger(__name__)
print("✓ Módulos importados com sucesso!")
print(f"✓ Conectando ao Postgres em {POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}")

/opt/conda/lib/python3.11/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


2026-05-26 21:04:43 | INFO | nl2sql2nl.config | Configurações carregadas com sucesso.
✓ Módulos importados com sucesso!
✓ Conectando ao Postgres em host.docker.internal:5432/mimic_fhir


## 2. Carregamento do dicionário de dados

In [3]:
# Carrega o arquivo de dicionário de dados
caminho_dicionario = projeto_raiz / "dic" / "dicionario_dados.yaml"

with open(caminho_dicionario, 'r', encoding='utf-8') as f:
    dicionario = yaml.safe_load(f)

print(f"✓ Dicionário de dados carregado de {caminho_dicionario}")
print(f"✓ Tabelas disponíveis: {len(dicionario['tabelas'])}")

# Exibe nomes das tabelas
nomes_tabelas = [tabela['nome'] for tabela in dicionario['tabelas']]
print(f"  Tabelas: {', '.join(nomes_tabelas)}")

✓ Dicionário de dados carregado de /home/jovyan/work/dic/dicionario_dados.yaml
✓ Tabelas disponíveis: 7
  Tabelas: organizacoes, localizacoes, pacientes, encontros, encontros_localizacoes, condicoes, procedimentos


## 3. Preparação do contexto do dicionário para o LLM

In [4]:
def formatar_dicionario_para_prompt(dicionario: dict) -> str:
    """
    Converte o dicionário de dados em formato legível para o LLM.
    Inclui informações sobre tabelas, colunas, tipos e relacionamentos.
    """
    linhas = []
    linhas.append("=== DICIONÁRIO DE DADOS MIMIC FHIR ===")
    linhas.append("")
    
    for tabela in dicionario['tabelas']:
        linhas.append(f"Tabela: {tabela['nome']}")
        linhas.append(f"Descrição: {tabela.get('descricao', 'N/A')}")
        linhas.append("Colunas:")
        
        for coluna in tabela['colunas']:
            nome = coluna['nome']
            tipo = coluna['tipo']
            descricao = coluna.get('descricao', 'N/A')
            chave_primaria = "[PK]" if coluna.get('chave_primaria') else ""
            chave_estrangeira = coluna.get('chave_estrangeira')
            fk_info = f"[FK: {chave_estrangeira}]" if chave_estrangeira else ""
            
            linhas.append(f"  - {nome} ({tipo}) {chave_primaria} {fk_info}")
            linhas.append(f"    {descricao}")
        
        linhas.append("")
    
    return "\n".join(linhas)

contexto_dicionario = formatar_dicionario_para_prompt(dicionario)
print("✓ Contexto do dicionário preparado para o LLM")
print(f"Tamanho do contexto: {len(contexto_dicionario)} caracteres")

✓ Contexto do dicionário preparado para o LLM
Tamanho do contexto: 3754 caracteres


In [5]:
def limpar_sql(sql_bruta: str) -> str:
    """
    Remove delimitadores de markdown (```sql ... ```) da SQL gerada pelo LLM.
    
    Args:
        sql_bruta: SQL potencialmente com delimitadores markdown.
    
    Returns:
        SQL limpa, sem delimitadores.
    """
    import re
    sql = sql_bruta.strip()
    sql = re.sub(r'^```(?:sql|SQL)?\n?', '', sql)
    sql = re.sub(r'\n?```$', '', sql)
    return sql.strip()


def gerar_sql_de_pergunta(pergunta: str, contexto_dicionario: str) -> str:
    """
    Converte uma pergunta em português para SQL usando o LLM.
    
    Args:
        pergunta: Pergunta em linguagem natural portuguesa.
        contexto_dicionario: Contexto com estrutura das tabelas do banco.
    
    Returns:
        String contendo a query SQL gerada.
    """
    logger.info(f"Gerando SQL para pergunta: {pergunta}")
    
    prompt = f"""Você é um especialista em SQL e base de dados MIMIC FHIR.
Sua tarefa é converter uma pergunta em linguagem natural para uma consulta SQL.

{contexto_dicionario}

Pergunta do usuário:
{pergunta}

Gere uma consulta SQL que responda a pergunta acima.
Observações importantes:
1. Use APENAS as tabelas e colunas descritas no dicionário acima.
2. Use inner join para relacionamentos entre tabelas.
3. Prefira select de colunas específicas em vez de select *.
4. Retorne APENAS a consulta SQL, sem explicações adicionais.
5. A query deve ser válida para PostgreSQL.

SQL:"""
    
    try:
        sql_gerada = chamar_modelo(prompt, modelo="gpt-5-nano")
        sql_limpa = limpar_sql(sql_gerada)
        logger.info(f"SQL gerada com sucesso: {sql_limpa[:100]}...")
        return sql_limpa
    except Exception as e:
        logger.error(f"Erro ao gerar SQL: {e}")
        raise

print("✓ Função gerar_sql_de_pergunta definida (com limpeza de markdown)")

✓ Função gerar_sql_de_pergunta definida (com limpeza de markdown)


In [6]:
def converter_resultado_para_resposta(pergunta: str, sql: str, resultados: list) -> str:
    """
    Converte os resultados da SQL em uma resposta em linguagem natural portuguesa.
    
    Args:
        pergunta: Pergunta original do usuário.
        sql: Consulta SQL que foi executada.
        resultados: Lista de dicionários com os resultados da SQL.
    
    Returns:
        String contendo a resposta em linguagem natural português.
    """
    logger.info(f"Convertendo {len(resultados)} linhas de resultado em resposta natural")
    
    # Formata os resultados para exibição
    resultados_str = ""
    if resultados:
        # Limite a quantidade de dados para não sobrecarregar o prompt
        primeiras_linhas = resultados[:5]
        for i, linha in enumerate(primeiras_linhas, 1):
            resultados_str += f"Linha {i}: {linha}\n"
        if len(resultados) > 5:
            resultados_str += f"... (total de {len(resultados)} linhas)\n"
    else:
        resultados_str = "(Nenhum resultado retornado)"
    
    prompt = f"""Você é um assistente que converte dados de banco de dados em respostas naturais em português.

Pergunta original do usuário:
{pergunta}

Consulta SQL executada:
{sql}

Resultados da consulta:
{resultados_str}

Gere uma resposta em linguagem natural portuguesa que:
1. Responda diretamente à pergunta original.
2. Resuma os dados de forma legível.
3. Seja concisa e clara.
4. Mencione a quantidade total de registros encontrados, se relevante.

Resposta:"""
    
    try:
        resposta = chamar_modelo(prompt, modelo="gpt-5-nano")
        logger.info(f"Resposta gerada com sucesso")
        return resposta.strip()
    except Exception as e:
        logger.error(f"Erro ao converter resultado: {e}")
        raise

print("✓ Função converter_resultado_para_resposta definida")

✓ Função converter_resultado_para_resposta definida


## 5. Execução do pipeline completo NL2SQL2NL

In [7]:
def executar_pipeline_nl2sql2nl(pergunta: str) -> dict:
    """
    Executa o pipeline completo: pergunta → SQL → execução → resposta natural.
    
    Args:
        pergunta: Pergunta em português do usuário.
    
    Returns:
        Dicionário com pergunta, sql_gerada, resultados_brutos e resposta_final.
    """
    resultado = {
        'pergunta': pergunta,
        'sql_gerada': None,
        'resultados_brutos': [],
        'resposta_final': None,
        'erro': None
    }
    
    try:
        logger.info("="*80)
        logger.info("INICIANDO PIPELINE NL2SQL2NL")
        logger.info("="*80)
        
        # Etapa 1: Gerar SQL
        logger.info("[1/4] Gerando SQL a partir da pergunta...")
        sql_gerada = gerar_sql_de_pergunta(pergunta, contexto_dicionario)
        resultado['sql_gerada'] = sql_gerada
        logger.info(f"[1/4] ✓ SQL gerada com sucesso")
        
        # Etapa 2: Executar SQL
        logger.info("[2/4] Executando SQL no Postgres...")
        resultados = executar_consulta(sql_gerada)
        resultado['resultados_brutos'] = resultados
        logger.info(f"[2/4] ✓ SQL executada, {len(resultados)} linhas retornadas")
        
        # Etapa 3: Converter para resposta natural
        logger.info("[3/4] Convertendo resultado para linguagem natural...")
        resposta_final = converter_resultado_para_resposta(pergunta, sql_gerada, resultados)
        resultado['resposta_final'] = resposta_final
        logger.info(f"[3/4] ✓ Resposta gerada com sucesso")
        
        logger.info("="*80)
        logger.info("PIPELINE NL2SQL2NL CONCLUÍDO COM SUCESSO")
        logger.info("="*80)
        
    except Exception as e:
        logger.error(f"ERRO no pipeline: {e}", exc_info=True)
        resultado['erro'] = str(e)
    
    return resultado

print("✓ Função executar_pipeline_nl2sql2nl definida")

✓ Função executar_pipeline_nl2sql2nl definida


## 6. Teste com a pergunta especificada

In [8]:
# Pergunta de teste especificada no projeto
pergunta_teste = "Quais são os períodos dos encontros, períodos das localizações e nomes das localizações do paciente com identificador 10000032?"

print(f"Pergunta de teste: {pergunta_teste}")
print(f"\nSQL esperada (referência):")
print("""select p.nome_familia, e.periodo_inicio, e.periodo_fim,
       el.periodo_inicio, el.periodo_fim, l.nome
from pacientes p inner join encontros e on p.id = e.paciente_id
inner join encontros_localizacoes el on el.encontro_id = e.id
inner join localizacoes l on el.localizacao_id = l.id
where p.identificador = '10000032'""")

Pergunta de teste: Quais são os períodos dos encontros, períodos das localizações e nomes das localizações do paciente com identificador 10000032?

SQL esperada (referência):
select p.nome_familia, e.periodo_inicio, e.periodo_fim,
       el.periodo_inicio, el.periodo_fim, l.nome
from pacientes p inner join encontros e on p.id = e.paciente_id
inner join encontros_localizacoes el on el.encontro_id = e.id
inner join localizacoes l on el.localizacao_id = l.id
where p.identificador = '10000032'


In [9]:
# Executa o pipeline com a pergunta de teste
resultado_pipeline = executar_pipeline_nl2sql2nl(pergunta_teste)

if resultado_pipeline['erro']:
    print(f"\n✗ ERRO NO PIPELINE: {resultado_pipeline['erro']}")
else:
    print("\n✓ Pipeline executado com sucesso!")

2026-05-26 21:04:44 | INFO | __main__ | ================================================================================
2026-05-26 21:04:44 | INFO | __main__ | INICIANDO PIPELINE NL2SQL2NL
2026-05-26 21:04:44 | INFO | __main__ | ================================================================================
2026-05-26 21:04:44 | INFO | __main__ | [1/4] Gerando SQL a partir da pergunta...
2026-05-26 21:04:44 | INFO | __main__ | Gerando SQL para pergunta: Quais são os períodos dos encontros, períodos das localizações e nomes das localizações do paciente com identificador 10000032?
2026-05-26 21:04:44 | INFO | nl2sql2nl.llm | Enviando pergunta ao modelo 'gpt-5-nano'.
2026-05-26 21:04:44 | INFO | nl2sql2nl.llm | Criando modelo LLM: gpt-5-nano (temperatura=0.0)
2026-05-26 21:04:55 | INFO | nl2sql2nl.llm | Resposta recebida com 480 caracteres.
2026-05-26 21:04:55 | INFO | __main__ | SQL gerada com sucesso: SELECT
  e.periodo_inicio AS periodo_encontro_inicio,
  e.periodo_fim AS periodo_enc

## 7. Exibição dos resultados

In [10]:
import pandas as pd
from IPython.display import display

print("\n" + "="*80)
print("RESULTADO FINAL DO PIPELINE NL2SQL2NL")
print("="*80)

print(f"\n📝 PERGUNTA DO USUÁRIO:")
print(f"{resultado_pipeline['pergunta']}")

print(f"\n🔍 SQL GERADA PELO LLM:")
print(f"{resultado_pipeline['sql_gerada']}")

print(f"\n📊 RESULTADOS BRUTOS (tabela):")
if resultado_pipeline['resultados_brutos']:
    df = pd.DataFrame(resultado_pipeline['resultados_brutos'])
    print(f"Total de linhas: {len(df)}")
    display(df)
else:
    print("Nenhum resultado retornado.")

print(f"\n💬 RESPOSTA FINAL (linguagem natural):")
print(f"{resultado_pipeline['resposta_final']}")

print("\n" + "="*80)


RESULTADO FINAL DO PIPELINE NL2SQL2NL

📝 PERGUNTA DO USUÁRIO:
Quais são os períodos dos encontros, períodos das localizações e nomes das localizações do paciente com identificador 10000032?

🔍 SQL GERADA PELO LLM:
SELECT
  e.periodo_inicio AS periodo_encontro_inicio,
  e.periodo_fim AS periodo_encontro_fim,
  il.periodo_inicio AS periodo_localizacao_inicio,
  il.periodo_fim AS periodo_localizacao_fim,
  l.nome AS localizacao_nome
FROM pacientes p
INNER JOIN encontros e ON e.paciente_id = p.id
INNER JOIN encontros_localizacoes il ON il.encontro_id = e.id
INNER JOIN localizacoes l ON l.id = il.localizacao_id
WHERE p.identificador = '10000032'
ORDER BY e.periodo_inicio, il.periodo_inicio;

📊 RESULTADOS BRUTOS (tabela):
Total de linhas: 9


,periodo_encontro_inicio,periodo_encontro_fim,periodo_localizacao_inicio,periodo_localizacao_fim,localizacao_nome
0,2180-05-06 22:23:00,2180-05-07 17:15:00,2180-05-06 19:17:00,2180-05-06 23:30:00,Emergency Department
1,2180-05-06 22:23:00,2180-05-07 17:15:00,2180-05-06 23:30:00,2180-05-07 17:21:27,Transplant
2,2180-06-26 18:27:00,2180-06-27 18:49:00,2180-06-26 15:54:00,2180-06-26 21:31:00,Emergency Department
3,2180-06-26 18:27:00,2180-06-27 18:49:00,2180-06-26 21:31:00,2180-06-27 18:49:12,Transplant
4,2180-07-23 12:35:00,2180-07-25 17:55:00,2180-07-22 16:24:00,2180-07-23 05:54:00,Emergency Department
5,2180-07-23 12:35:00,2180-07-25 17:55:00,2180-07-23 14:00:00,2180-07-23 23:50:47,Medical Intensive Care Unit (MICU)
6,2180-07-23 12:35:00,2180-07-25 17:55:00,2180-07-23 23:50:47,2180-07-24 19:52:58,Transplant
7,2180-08-05 23:44:00,2180-08-07 17:50:00,2180-08-05 20:58:00,2180-08-06 01:44:00,Emergency Department
8,2180-08-05 23:44:00,2180-08-07 17:50:00,2180-08-06 01:44:00,2180-08-07 17:50:44,Transplant



💬 RESPOSTA FINAL (linguagem natural):
Para o paciente com identificador 10000032, foram encontrados 9 registros. Abaixo estão os 5 primeiros, com os períodos dos encontros, os períodos das localizações e o nome das localizações:

1) Encontro: 6/5/2180 22:23 – 7/5/2180 17:15
   Localização: Emergency Department
   Período da localização: 6/5/2180 19:17 – 6/5/2180 23:30

2) Encontro: 6/5/2180 22:23 – 7/5/2180 17:15
   Localização: Transplant
   Período da localização: 6/5/2180 23:30 – 7/5/2180 17:21:27

3) Encontro: 26/6/2180 18:27 – 27/6/2180 18:49
   Localização: Emergency Department
   Período da localização: 26/6/2180 15:54 – 26/6/2180 21:31

4) Encontro: 26/6/2180 18:27 – 27/6/2180 18:49
   Localização: Transplant
   Período da localização: 26/6/2180 21:31 – 27/6/2180 18:49:12

5) Encontro: 23/7/2180 12:35 – 25/7/2180 17:55
   Localização: Emergency Department
   Período da localização: 22/7/2180 16:24 – 23/7/2180 05:54

Observação: existem 4 registros adicionais que não estão exib

## 8. Teste com pergunta customizada (opcional)

In [11]:
# Célula opcional para testar com outras perguntas
pergunta_customizada = "Quais são os nomes dos procedimentos realizados no paciente com identificador 10000032? Retonar as datas dos eventos"

print(f"Testando pergunta customizada: {pergunta_customizada}\n")

resultado_custom = executar_pipeline_nl2sql2nl(pergunta_customizada)

if not resultado_custom['erro']:
    print(f"\n🔍 SQL GERADA:")
    print(resultado_custom['sql_gerada'])
    
    print(f"\n📊 RESULTADOS ({len(resultado_custom['resultados_brutos'])} linhas):")
    if resultado_custom['resultados_brutos']:
        df = pd.DataFrame(resultado_custom['resultados_brutos'])
        display(df)
    
    print(f"\n💬 RESPOSTA:")
    print(resultado_custom['resposta_final'])
else:
    print(f"✗ ERRO: {resultado_custom['erro']}")

Testando pergunta customizada: Quais são os nomes dos procedimentos realizados no paciente com identificador 10000032? Retonar as datas dos eventos

2026-05-26 21:05:13 | INFO | __main__ | ================================================================================
2026-05-26 21:05:13 | INFO | __main__ | INICIANDO PIPELINE NL2SQL2NL
2026-05-26 21:05:13 | INFO | __main__ | ================================================================================
2026-05-26 21:05:13 | INFO | __main__ | [1/4] Gerando SQL a partir da pergunta...
2026-05-26 21:05:13 | INFO | __main__ | Gerando SQL para pergunta: Quais são os nomes dos procedimentos realizados no paciente com identificador 10000032? Retonar as datas dos eventos
2026-05-26 21:05:13 | INFO | nl2sql2nl.llm | Enviando pergunta ao modelo 'gpt-5-nano'.
2026-05-26 21:05:13 | INFO | nl2sql2nl.llm | Criando modelo LLM: gpt-5-nano (temperatura=0.0)
2026-05-26 21:05:18 | INFO | nl2sql2nl.llm | Resposta recebida com 257 caracteres.
2026-05-26

,nome_procedimento,data_hora_procedimento
0,Percutaneous abdominal drainage,2180-05-07
1,Percutaneous abdominal drainage,2180-06-27
2,Percutaneous abdominal drainage,2180-08-06



💬 RESPOSTA:
Para o paciente com identificador 10000032, o procedimento registrado é o mesmo em três momentos:

- Nome do procedimento: Percutaneous abdominal drainage
- Datas dos eventos: 07/05/2180; 27/06/2180; 06/08/2180

Total de registros: 3


## 9. Resumo do pipeline NL2SQL2NL

Este notebook demonstra o fluxo completo:

1. ✅ **Carregamento de variáveis do .env** — configurações do Postgres e OpenAI
2. ✅ **Carregamento do dicionário de dados** — `/dic/dicionario_dados.yaml`
3. ✅ **Geração de SQL** — LLM transforma pergunta em português para SQL
4. ✅ **Execução no Postgres** — SQL é executada no banco de dados
5. ✅ **Conversão para linguagem natural** — Resultados são transformados em resposta em português
6. ✅ **Exibição de resultados** — Pergunta, SQL gerada, dados brutos e resposta final

### Características:
- Código em português com docstrings e comentários
- Logs para fluxo normal, erros e falhas
- Uso de variáveis do `.env`
- Simples chamadas com LangChain e OpenAI (sem LangGraph)
- Tratamento de erros robusto